# CSCI 6970 – Classifying Diagnosed Diabetes Using NHANES Data

## 1. Project Environment Setup

### 1.1 Project Overview

This project aims to develop machine learning classification models to identify diagnosed diabetes using demographic, lifestyle, and health indicators from the National Health and Nutrition Examination Survey (NHANES) August 2021–August 2023 dataset.

The project uses Python, Pandas, NumPy, Scikit-learn, and JupyterLab for data collection, preprocessing, statistical analysis, and machine learning.

In [1]:
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn

print("Python:", sys.version)
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("Scikit-learn:", sklearn.__version__)

print("\nCSCI6970 Environment Ready!")

Python: 3.9.6 (default, Aug 25 2026, 21:26:21) 
[Clang 21.0.0 (clang-2100.3.34.2)]
Pandas: 2.3.3
NumPy: 2.0.2
Scikit-learn: 1.6.1

CSCI6970 Environment Ready!


### 1.2 Project Directory Structure

The project is organized into separate directories for raw data, processed data, notebooks, outputs, and scripts to maintain a reproducible workflow.

In [2]:
from pathlib import Path

# Project directory
PROJECT_DIR = Path.cwd().parent

# Data directories
RAW_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Project directory:", PROJECT_DIR)
print("Raw data directory:", RAW_DIR)
print("Processed data directory:", PROCESSED_DIR)

Project directory: /Users/iamajaygaur/Downloads/SEM III/Data Science/CSCI6970
Raw data directory: /Users/iamajaygaur/Downloads/SEM III/Data Science/CSCI6970/data/raw
Processed data directory: /Users/iamajaygaur/Downloads/SEM III/Data Science/CSCI6970/data/processed


## 2. NHANES Data Collection

### 2.1 NHANES Dataset Selection

The study uses publicly available NHANES August 2021–August 2023 data released by the Centers for Disease Control and Prevention (CDC).

### 2.2 Identification of Required NHANES Datasets

Eight NHANES datasets were selected for this study: demographics, diabetes, body measurements, blood pressure examination, blood pressure questionnaire, physical activity, smoking, and alcohol use.

### 2.3 Downloading NHANES XPT Files

In [3]:
import requests
from pathlib import Path
import time

# NHANES August 2021–August 2023
BASE_URL = (
    "https://wwwn.cdc.gov/Nchs/Data/Nhanes/"
    "Public/2021/DataFiles/"
)

DATA_FILES = [
    "DEMO_L",
    "DIQ_L",
    "BMX_L",
    "BPXO_L",
    "BPQ_L",
    "PAQ_L",
    "SMQ_L",
    "ALQ_L"
]

RAW_DIR.mkdir(parents=True, exist_ok=True)

download_results = []

for file_name in DATA_FILES:

    file_url = f"{BASE_URL}{file_name}.XPT"
    file_path = RAW_DIR / f"{file_name}.XPT"

    try:
        print(f"Downloading {file_name}...", end=" ")

        response = requests.get(
            file_url,
            timeout=120
        )

        response.raise_for_status()

        # Save original XPT file
        file_path.write_bytes(response.content)

        file_size = file_path.stat().st_size / 1024

        download_results.append({
            "Dataset": file_name,
            "Status": "Downloaded",
            "Size (KB)": round(file_size, 2)
        })

        print(f"Success ({file_size:.2f} KB)")

    except Exception as error:

        download_results.append({
            "Dataset": file_name,
            "Status": "Failed",
            "Size (KB)": 0
        })

        print(f"Failed: {error}")

    time.sleep(1)

# Display download summary
import pandas as pd

download_summary = pd.DataFrame(download_results)

display(download_summary)

/Users/iamajaygaur/Downloads/SEM III/Data Science/CSCI6970/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


,Dataset,Status,Size (KB)
0,DEMO_L,Downloaded,2521.64
1,DIQ_L,Downloaded,827.73
2,BMX_L,Downloaded,1526.56
3,BPXO_L,Downloaded,680.39
4,BPQ_L,Downloaded,400.08
5,PAQ_L,Downloaded,399.92
6,SMQ_L,Downloaded,635.86
7,ALQ_L,Downloaded,447.58


### 2.4 Raw Data Storage and Verification

The downloaded XPT files are stored in the data/raw directory. Their presence and file sizes are verified to confirm successful data collection.

In [4]:
print("NHANES RAW DATA VERIFICATION")
print("-" * 45)

for file in sorted(RAW_DIR.glob("*.XPT")):
    print(
        f"{file.name:15} "
        f"{file.stat().st_size / 1024:.2f} KB"
    )

print("-" * 45)
print("Total files:", len(list(RAW_DIR.glob("*.XPT"))))

NHANES RAW DATA VERIFICATION
---------------------------------------------
ALQ_L.XPT       447.58 KB
BMX_L.XPT       1526.56 KB
BPQ_L.XPT       400.08 KB
BPXO_L.XPT      680.39 KB
DEMO_L.XPT      2521.64 KB
DIQ_L.XPT       827.73 KB
PAQ_L.XPT       399.92 KB
SMQ_L.XPT       635.86 KB
---------------------------------------------
Total files: 8


## 3. NHANES Dataset Inspection

### 3.1 Loading NHANES XPT Files

The downloaded NHANES datasets are loaded using Pandas to examine their structure.

### 3.2 Inspecting Dataset Dimensions and Variables

The number of rows, columns, and available variables is examined for each dataset.

### 3.3 Verifying Participant Identifier (SEQN)

The SEQN variable is checked across all datasets because it serves as the common participant identifier for subsequent data integration.

In [5]:
import pandas as pd
from pathlib import Path

print("NHANES DATASET INSPECTION")
print("=" * 60)

dataset_summary = []

for file_path in sorted(RAW_DIR.glob("*.XPT")):
    df = pd.read_sas(file_path, format="xport")

    dataset_summary.append({
        "Dataset": file_path.stem,
        "Rows": df.shape[0],
        "Columns": df.shape[1],
        "SEQN Available": "SEQN" in df.columns
    })

    print(f"\n{file_path.stem}")
    print("Shape:", df.shape)
    print("First 10 columns:", list(df.columns[:10]))

summary_df = pd.DataFrame(dataset_summary)

print("\n" + "=" * 60)
display(summary_df)

NHANES DATASET INSPECTION

ALQ_L
Shape: (6337, 9)
First 10 columns: ['SEQN', 'ALQ111', 'ALQ121', 'ALQ130', 'ALQ142', 'ALQ270', 'ALQ280', 'ALQ151', 'ALQ170']

BMX_L
Shape: (8860, 22)
First 10 columns: ['SEQN', 'BMDSTATS', 'BMXWT', 'BMIWT', 'BMXRECUM', 'BMIRECUM', 'BMXHEAD', 'BMIHEAD', 'BMXHT', 'BMIHT']

BPQ_L
Shape: (8501, 6)
First 10 columns: ['SEQN', 'BPQ020', 'BPQ030', 'BPQ150', 'BPQ080', 'BPQ101D']

BPXO_L
Shape: (7801, 12)
First 10 columns: ['SEQN', 'BPAOARM', 'BPAOCSZ', 'BPXOSY1', 'BPXODI1', 'BPXOSY2', 'BPXODI2', 'BPXOSY3', 'BPXODI3', 'BPXOPLS1']

DEMO_L
Shape: (11933, 27)
First 10 columns: ['SEQN', 'SDDSRVYR', 'RIDSTATR', 'RIAGENDR', 'RIDAGEYR', 'RIDAGEMN', 'RIDRETH1', 'RIDRETH3', 'RIDEXMON', 'RIDEXAGM']

DIQ_L
Shape: (11744, 9)
First 10 columns: ['SEQN', 'DIQ010', 'DID040', 'DIQ160', 'DIQ180', 'DIQ050', 'DID060', 'DIQ060U', 'DIQ070']

PAQ_L
Shape: (8153, 8)
First 10 columns: ['SEQN', 'PAD790Q', 'PAD790U', 'PAD800', 'PAD810Q', 'PAD810U', 'PAD820', 'PAD680']

SMQ_L
Shape: (9015, 9

,Dataset,Rows,Columns,SEQN Available
0,ALQ_L,6337,9,True
1,BMX_L,8860,22,True
2,BPQ_L,8501,6,True
3,BPXO_L,7801,12,True
4,DEMO_L,11933,27,True
5,DIQ_L,11744,9,True
6,PAQ_L,8153,8,True
7,SMQ_L,9015,9,True


## 4. Diabetes Target Variable Preparation

### 4.1 Loading the Diabetes Questionnaire Dataset

The DIQ_L dataset is loaded to identify the variable used as the diabetes classification target.

### 4.2 Identification of Diabetes Target Variable (DIQ010)

DIQ010 records whether a participant has been told by a doctor or health professional that they have diabetes.

### 4.3 Inspecting Diabetes Response Categories

The response counts and percentages are examined, including valid, borderline, refused, and missing responses.

In [6]:
# Load the diabetes questionnaire dataset
diq_df = pd.read_sas(
    RAW_DIR / "DIQ_L.XPT",
    format="xport"
)

# Check the target variable
print("DIABETES TARGET VARIABLE: DIQ010")
print("=" * 50)

print("\nResponse counts:")
print(diq_df["DIQ010"].value_counts(dropna=False).sort_index())

print("\nResponse percentages:")
print(
    (diq_df["DIQ010"].value_counts(normalize=True, dropna=False) * 100)
    .sort_index()
    .round(2)
)

print("\nMissing target values:")
print(diq_df["DIQ010"].isna().sum())

DIABETES TARGET VARIABLE: DIQ010

Response counts:
DIQ010
1.0     1081
2.0    10371
3.0      284
9.0        4
NaN        4
Name: count, dtype: int64

Response percentages:
DIQ010
1.0     9.20
2.0    88.31
3.0     2.42
9.0     0.03
NaN     0.03
Name: proportion, dtype: float64

Missing target values:
4


### 4.4 Creating the Binary Diabetes Target Variable

The binary target variable DIABETES is created using the following coding:

- 1 = Yes, diagnosed with diabetes.
- 0 = No, not diagnosed with diabetes.

Borderline, refused, and missing target responses are excluded from the primary binary classification dataset.

### 4.5 Examining Target Distribution and Duplicate IDs

The resulting class distribution and participant identifier uniqueness are verified.

In [7]:
# Create a copy of the diabetes questionnaire dataset
diabetes_df = diq_df.copy()

# Keep only valid Yes/No responses
diabetes_df = diabetes_df[
    diabetes_df["DIQ010"].isin([1.0, 2.0])
].copy()

# Create binary target variable
# 1 = Yes, diagnosed with diabetes
# 0 = No, not diagnosed with diabetes
diabetes_df["DIABETES"] = diabetes_df["DIQ010"].map({
    1.0: 1,
    2.0: 0
})

# Display target distribution
print("BINARY DIABETES TARGET SUMMARY")
print("=" * 45)

print("Total participants:", len(diabetes_df))
print("\nTarget distribution:")
print(diabetes_df["DIABETES"].value_counts().sort_index())

print("\nTarget percentages:")
print(
    (diabetes_df["DIABETES"].value_counts(normalize=True) * 100)
    .sort_index()
    .round(2)
)

print("\nDuplicate participant IDs:", diabetes_df["SEQN"].duplicated().sum())

# Keep only participant ID and target variable
diabetes_target = diabetes_df[["SEQN", "DIABETES"]].copy()

display(diabetes_target.head())

BINARY DIABETES TARGET SUMMARY
Total participants: 11452

Target distribution:
DIABETES
0    10371
1     1081
Name: count, dtype: int64

Target percentages:
DIABETES
0    90.56
1     9.44
Name: proportion, dtype: float64

Duplicate participant IDs: 0


,SEQN,DIABETES
0,130378.0,0
1,130379.0,0
2,130380.0,1
3,130381.0,0
4,130382.0,0


## 5. Adult Population Filtering and Initial Data Merging

### 5.1 Loading the Demographic Dataset

The DEMO_L dataset is loaded to obtain participant demographic information.

### 5.2 Inspecting Age Variable (RIDAGEYR)

The age variable is examined using descriptive statistics and missing-value checks.

### 5.3 Filtering Adult Participants (Age 20 and Above)

The study population is restricted to participants aged 20 years and older.

In [8]:
# Load NHANES demographic dataset
demo_df = pd.read_sas(
    RAW_DIR / "DEMO_L.XPT",
    format="xport"
)

print("DEMOGRAPHIC DATA INSPECTION")
print("=" * 45)

print("Dataset shape:", demo_df.shape)

print("\nAge variable information:")
print(demo_df["RIDAGEYR"].describe())

print("\nMissing age values:")
print(demo_df["RIDAGEYR"].isna().sum())

# Count participants aged 20 or older
adult_demo = demo_df[
    demo_df["RIDAGEYR"] >= 20
].copy()

print("\nParticipants aged 20 or older:", len(adult_demo))

# Check participant ID uniqueness
print(
    "Duplicate SEQN IDs:",
    adult_demo["SEQN"].duplicated().sum()
)

DEMOGRAPHIC DATA INSPECTION
Dataset shape: (11933, 27)

Age variable information:
count    1.193300e+04
mean     3.831786e+01
std      2.560199e+01
min      5.397605e-79
25%      1.300000e+01
50%      3.700000e+01
75%      6.200000e+01
max      8.000000e+01
Name: RIDAGEYR, dtype: float64

Missing age values:
0

Participants aged 20 or older: 7809
Duplicate SEQN IDs: 0


### 5.4 Merging Adult Demographics with Diabetes Target

The adult demographic dataset is merged with the binary diabetes target using SEQN. An inner join retains participants with both demographic information and a valid diabetes response.

### 5.5 Verifying the Adult Diabetes Dataset

The merged dataset dimensions, diabetes distribution, age range, and duplicate participant IDs are examined.

In [9]:
# Merge adult demographics with diabetes target
adult_diabetes_df = pd.merge(
    adult_demo,
    diabetes_target,
    on="SEQN",
    how="inner",
    validate="one_to_one"
)

print("ADULT DIABETES DATASET")
print("=" * 50)

print("Dataset shape:", adult_diabetes_df.shape)
print("Total adult participants:", len(adult_diabetes_df))

print("\nDiabetes distribution:")
print(adult_diabetes_df["DIABETES"].value_counts().sort_index())

print("\nDiabetes percentages:")
print(
    (adult_diabetes_df["DIABETES"].value_counts(normalize=True) * 100)
    .sort_index()
    .round(2)
)

print("\nAge range:")
print(
    adult_diabetes_df["RIDAGEYR"].min(),
    "to",
    adult_diabetes_df["RIDAGEYR"].max()
)

print("\nDuplicate participant IDs:",
      adult_diabetes_df["SEQN"].duplicated().sum())

display(adult_diabetes_df[["SEQN", "RIDAGEYR", "DIABETES"]].head())

ADULT DIABETES DATASET
Dataset shape: (7536, 28)
Total adult participants: 7536

Diabetes distribution:
DIABETES
0    6466
1    1070
Name: count, dtype: int64

Diabetes percentages:
DIABETES
0    85.8
1    14.2
Name: proportion, dtype: float64

Age range:
20.0 to 80.0

Duplicate participant IDs: 0


,SEQN,RIDAGEYR,DIABETES
0,130378.0,43.0,0
1,130379.0,66.0,0
2,130380.0,44.0,1
3,130384.0,43.0,0
4,130385.0,65.0,0


## 6. Additional NHANES Dataset Integration

### 6.1 Integrating Body Measurements (BMX_L)

The BMX_L dataset contains body measurement information collected during NHANES examinations.

For this project, we will examine variables such as:

- BMXBMI – Body Mass Index (BMI)
- BMXWT – Weight measurement
- BMXHT – Standing height
- BMXWAIST – Waist circumference

These variables will be considered as potential predictors of diagnosed diabetes.

The dataset will be integrated with the existing adult diabetes dataset using the participant identifier SEQN.

In [15]:
# Load NHANES body measurements dataset
bmx_df = pd.read_sas(
    RAW_DIR / "BMX_L.XPT",
    format="xport"
)

print("BODY MEASUREMENTS DATASET INSPECTION")
print("=" * 55)

print("Dataset shape:", bmx_df.shape)

print("\nAvailable variables:")
print(bmx_df.columns.tolist())

print("\nSelected body measurement variables:")

bmx_variables = [
    "SEQN",
    "BMXBMI",
    "BMXWT",
    "BMXHT",
    "BMXWAIST"
]

display(bmx_df[bmx_variables].head())

print("\nMissing values in selected variables:")
display(bmx_df[bmx_variables].isna().sum())

print(
    "\nDuplicate participant IDs:",
    bmx_df["SEQN"].duplicated().sum()
)

BODY MEASUREMENTS DATASET INSPECTION
Dataset shape: (8860, 22)

Available variables:
['SEQN', 'BMDSTATS', 'BMXWT', 'BMIWT', 'BMXRECUM', 'BMIRECUM', 'BMXHEAD', 'BMIHEAD', 'BMXHT', 'BMIHT', 'BMXBMI', 'BMDBMIC', 'BMXLEG', 'BMILEG', 'BMXARML', 'BMIARML', 'BMXARMC', 'BMIARMC', 'BMXWAIST', 'BMIWAIST', 'BMXHIP', 'BMIHIP']

Selected body measurement variables:


,SEQN,BMXBMI,BMXWT,BMXHT,BMXWAIST
0,130378.0,27.0,86.9,179.5,98.3
1,130379.0,33.5,101.8,174.2,114.7
2,130380.0,29.7,69.4,152.9,93.5
3,130381.0,23.8,34.3,120.1,70.4
4,130382.0,NaN,13.6,NaN,NaN



Missing values in selected variables:


SEQN          0
BMXBMI      389
BMXWT       106
BMXHT       361
BMXWAIST    670
dtype: int64


Duplicate participant IDs: 0
